# as-strided-noncontig-source — faded example 3: Sliding window with step 2 over a 1-D tensor

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-noncontig-source`. Running the beacon reports progress on the `Numpy: Applied patterns and advanced` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `as-strided-noncontig-source`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

A sliding window doesn't have to advance by one element per row. With a *hop* of `k`, row `i` starts at source position `i*k`. The within-window step is still 1. So for `x.stride(0) == s`, the output stride is `(k * s, s)` and there are `(N - w) // k + 1` windows.

## Faded exercise 3

### Faded — strided sliding window (hop = 2)

Implement `hop_window(x, w, k)`: a zero-copy `(num, w)` view where row `i` is `x[i*k : i*k + w]`, i.e. windows of length `w` advancing `k` elements each.

The number of rows and the `as_strided` call are written for you. Complete the **stride** tuple.

**Fill in:** The 2-tuple stride: advance k source elements per output row, 1 per output column.

In [ ]:
def hop_window(x: Tensor, w: int, k: int) -> Tensor:
    n = x.shape[0]
    s = x.stride(0)
    num = (n - w) // k + 1
    stride = None  # TODO: stride that hops k elements per row, 1 per column
    return t.as_strided(x, size=(num, w), stride=stride)


def _test():
    x = t.arange(10, dtype=t.float32)
    w, k = 3, 2
    out = hop_window(x, w, k)
    num = (10 - w) // k + 1
    assert tuple(out.shape) == (num, w), out.shape
    # build the reference explicitly
    ref = t.stack([x[i * k : i * k + w] for i in range(num)])
    assert t.equal(out, ref), (out, ref)
    assert out.data_ptr() == x.data_ptr()
    assert out[0].tolist() == [0.0, 1.0, 2.0]
    assert out[1].tolist() == [2.0, 3.0, 4.0]


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def hop_window(x: Tensor, w: int, k: int) -> Tensor:
    n = x.shape[0]
    s = x.stride(0)
    num = (n - w) // k + 1
    stride = (k * s, s)
    return t.as_strided(x, size=(num, w), stride=stride)
```
</details>